# Упражнение: Разбор лога ReAct: повторы действий, зацикливание и эффективность

## Задание

В `input.csv` — логи 24 запусков ReAct-агента (`run_id, line_no, text`): каждая строка лога лежит в отдельной
строке таблицы, **порядок строк в файле перемешан** — восстановите его по `line_no` внутри каждого `run_id`.
По краям `text` могут быть лишние пробелы — уберите их (`strip`).

Строки лога бывают такими:
- `Thought: ...` — рассуждение;
- `Action: имя[аргумент]` — действие; **корректный** формат — регулярное выражение `^Action: (\w+)\[(.*)\]$`
  (например, `Action: search[столица Франции]`); строки `Action: search Париж` и `Action: lookup(Париж)`
  — это **ошибки формата**;
- `Observation: ...` — наблюдение;
- `Final Answer: ...` — финальный ответ.

Нормализуйте действие: имя и аргумент — в нижний регистр, пробелы по краям аргумента убрать, внутренние
пробелы схлопнуть (`' '.join(text.split())`). Два действия одинаковы, если одинаковы нормализованные пары
`(имя, аргумент)`.

Для каждого запуска посчитайте:
- `steps` — число строк, начинающихся с `Action:` (включая ошибочные);
- `repeats` — сколько корректных действий **уже встречались раньше** в этом запуске (первое появление не
  считается: три одинаковых действия дают `repeats = 2`);
- `looped` — 1, если какое-то корректное действие встретилось **3 и более раз**, иначе 0;
- `format_errors` — число строк `Action:` с неверным форматом;
- `orphan_obs` — число строк `Observation:`, которым **не предшествует** строка `Action:`
  (предыдущая строка запуска — не `Action:`): такое наблюдение модель выдумала сама;
- `finished` — 1, если в запуске есть строка `Final Answer:`, иначе 0;
- `efficiency` — число **различных** корректных действий, делённое на `steps` (0, если `steps = 0`),
  округлить до 4 знаков.

## Подсказки

- Сначала `sort_values(['run_id', 'line_no'])`, и только потом разбирайте строки подряд: понятие «предыдущая
  строка» имеет смысл лишь в восстановленном порядке.
- Для повторов ведите множество уже виденных пар `(имя, аргумент)` и счётчик `Counter`: `repeats` — сумма
  `(count - 1)`, `looped` — наличие `count >= 3`.
- `Action: search Париж` — это строка `Action:`, значит она учитывается в `steps` и `format_errors`, но не в
  повторах и не в различных действиях.

## Ожидаемый формат выходного файла

CSV с колонками `run_id, steps, repeats, looped, format_errors, orphan_obs, finished, efficiency` — 24 строки,
по одной на запуск, отсортированные по `run_id`; `efficiency` округлить до 4 знаков.

## Проверка

Сохраните свой результат в `my_answer.csv` (в папке упражнения) и выполните из корня репозитория:

```bash
python tools/check_exercise.py exercises/react_agents
```

Эталонное решение с пояснениями — `solution.py` (загляните в него только после своей попытки).
Верное решение портал отметит сам и откроет разбор (если он запущен через `python start.py`).

## Данные

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv('input.csv')
print(df.shape)
df.head()

## Решение

In [ ]:
# Ваше решение. Итог запишите в DataFrame `result` — в формате, описанном в задании.

result = None

## Проверка

In [ ]:
# Проверка: сравнение result с эталоном (запускайте после своего решения).
# Сохраняет result в my_answer.csv, подсказывает вероятную причину ошибки, верное решение отмечает в портале.
import sys
from pathlib import Path

root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'tools' / 'check_exercise.py').exists())
sys.path.insert(0, str(root / 'tools'))
from check_exercise import check_result

check_result(result)

---
Эталонное решение с пояснениями — в файле `solution.py`. Загляните в него после своей попытки.